## BUILD LCEL CHAIN

In [2]:
%pip install -U langchain langchain-core langchain-groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


## To Access The .env Variables

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
api_key = os.getenv("GROQ_API_KEY")
# import the required libraries
# to access env variables
import os
# to load api keys
from dotenv import load_dotenv
# to create reusable chat prompts
from langchain_core.prompts import ChatPromptTemplate
# StrOutputParser converting llm output to simple string
from langchain_core.output_parsers import StrOutputParser
# allows langchain to connect with groq
from langchain_groq import ChatGroq

In [4]:
llm = ChatGroq(
    model ="openai/gpt-oss-120b",
    temperature = 0
)
print("llm is ready to use")

llm is ready to use


In [5]:
result = llm.invoke("What is the capital of france?")
print(result)

content='The capital of France is **Paris**.' additional_kwargs={'reasoning_content': 'The user asks: "What is the capital of france?" Simple factual question. Answer: Paris. Provide answer.'} response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 78, 'total_tokens': 120, 'completion_time': 0.086505165, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.003090308, 'prompt_tokens_details': None, 'queue_time': 0.350722045, 'total_time': 0.089595473}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8ea50c0161', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0fdaa-2a13-71f0-a50a-ffb8f8be0e82-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 78, 'output_tokens': 42, 'total_tokens': 120, 'output_token_details': {'reasoning': 24}}


In [6]:
prompt = ChatPromptTemplate.from_template(
    """
    Summarise the following text 
    in a simple language
    Text:
    {text}
    """
)

## Test to Prompt

In [7]:
formatted_prompt = prompt.invoke({
    "text":"Hey Pretty!"
})
print(formatted_prompt)

messages=[HumanMessage(content='\n    Summarise the following text \n    in a simple language\n    Text:\n    Hey Pretty!\n    ', additional_kwargs={}, response_metadata={})]


## BUILD A LCEL PIPELINE

In [8]:
chain = llm|prompt

In [9]:
parser = StrOutputParser()
chain = prompt|llm|parser

In [10]:
result = chain.invoke(
    {
        "text":"BMW Car"
    }
)
print(result)

**Simple summary:**  
BMW is a German car company that makes high‑quality, luxury vehicles known for good performance and stylish design.


## BUILD DATA INGESTION PIPELINE

In [11]:
%pip install -U langchain-community langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers python-dotenv


  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 1.1.2
    Uninstalling langchain-text-splitters-1.1.2:
      Successfully uninstalled langchain-text-splitters-1.1.2
Note: you may need to restart the kernel to use updated packages.


In [12]:
#Import the Required Libraries
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

C:\Users\asus\AppData\Local\Temp\ipykernel_22688\2054642680.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [13]:
#Check the documents Available 
documents_path = Path("documents")
files = list(documents_path.glob("*.txt"))
print(f"Documents found: {len(files)}")
for file in files:
    print(file.name)

Documents found: 10
Advance coding questions and Answers.txt
css.txt
dbms.txt
DSA levels 1to 3 .txt
GenAI.txt
Html.txt
java.txt
JS.txt
ML.txt
python.txt


In [14]:
%pip install chardet

Note: you may need to restart the kernel to use updated packages.


In [15]:
#loading all the documents

all_documents=[]
for file in files:
    loader=TextLoader(str(file), encoding="utf-8")
    docs=loader.load()
    all_documents.extend(docs)
print("Total documents:",all_documents)

Total documents: [Document(metadata={'source': 'documents\\Advance coding questions and Answers.txt'}, page_content="# Advanced Coding Questions and Answers\n\n### 1. What is time complexity?\n\n**Answer:** Time complexity describes how the running time of an algorithm grows as the input size increases.\n\n### 2. What is space complexity?\n\n**Answer:** Space complexity describes how much additional memory an algorithm requires as the input size grows.\n\n### 3. What is Big-O notation?\n\n**Answer:** Big-O notation describes the upper-bound growth rate of an algorithm's time or space requirements.\n\n### 4. What is recursion?\n\n**Answer:** Recursion is a programming technique where a function calls itself to solve smaller instances of a problem.\n\n### 5. What is dynamic programming?\n\n**Answer:** Dynamic programming solves complex problems by dividing them into overlapping subproblems and storing their results for reuse.\n\n### 6. What is memoization?\n\n**Answer:** Memoization stor

In [16]:
#Splitting the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

In [17]:
chunks = text_splitter.split_documents(all_documents)
print("Original documents:",len(all_documents))
print("Total chunks:",len(chunks))

Original documents: 10
Total chunks: 188


In [18]:
for i,chunk in enumerate(chunks):
    print(f"CHUNK{i+1}")
    print("source:",chunk.metadata["source"])
    print(chunk.page_content)

CHUNK1
source: documents\Advance coding questions and Answers.txt
# Advanced Coding Questions and Answers

### 1. What is time complexity?

**Answer:** Time complexity describes how the running time of an algorithm grows as the input size increases.

### 2. What is space complexity?

**Answer:** Space complexity describes how much additional memory an algorithm requires as the input size grows.

### 3. What is Big-O notation?

**Answer:** Big-O notation describes the upper-bound growth rate of an algorithm's time or space requirements.
CHUNK2
source: documents\Advance coding questions and Answers.txt
### 4. What is recursion?

**Answer:** Recursion is a programming technique where a function calls itself to solve smaller instances of a problem.

### 5. What is dynamic programming?

**Answer:** Dynamic programming solves complex problems by dividing them into overlapping subproblems and storing their results for reuse.

### 6. What is memoization?

**Answer:** Memoization stores the res

In [19]:
#create embeddings models
from langchain_huggingface import HuggingFaceEmbeddings
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 864.36it/s]


In [20]:
# Build FAISS Store
vector_store = FAISS.from_documents(
    chunks,
    embeddings_model
)
print("Vector store created successfully")

Vector store created successfully


In [21]:
from sentence_transformers.util import similarity
query = "python problems"

results=vector_store.similarity_search(
    query,
    k=2
)

In [ ]:
for i,result in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print("Source:",result.metadata["source"])
    print(result.page_content)


--- Chunk 1 ---
Source: documents\python.txt
---

## 2. Beginner Level – Build the Foundation

Students should **not start with frameworks immediately**. First learn Python fundamentals.

### Variables

Variables store values:

```python
name = "Rahul"
age = 21
marks = 85.5
```

### Data Types

Learn:

* `int`
* `float`
* `str`
* `bool`
* `list`
* `tuple`
* `set`
* `dict`
* `None`

Example:

```python
student = {
    "name": "Rahul",
    "age": 21,
    "course": "BTech"
}
```

### Operators

Practice:

--- Chunk 2 ---
Source: documents\python.txt
### Operators

Practice:

* Arithmetic
* Comparison
* Logical
* Assignment
* Membership
* Identity

### Conditional Statements

```python
marks = 75

if marks >= 40:
    print("Pass")
else:
    print("Fail")
```

### Loops

Loops allow repeated execution.

```python
for i in range(1, 6):
    print(i)
```

Students should practice `for`, `while`, `break`, `continue`, and `pass`.

---

# 3. Functions – The Next Step

Functions make programs reu

: 